In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from spx_history import *
from intra_adj import *

- for one symbol first

In [3]:
start_dt = dt.date(2021,1,1)
end_dt = dt.date(2026,2,1)

- one symbol only

In [4]:
# all dividends data
ALL_DIV = pd.read_csv('data_by_date/all_dividends_till_20260201.csv')
ALL_DIV['Date'] = pd.to_datetime(ALL_DIV['Date']).dt.date

# read holidays
th_us = pd.read_csv('us_holidays.csv')
th_us['date'] = pd.to_datetime(th_us['date'], dayfirst=True).dt.date
US_HOLS = th_us['date'].unique().tolist()

In [5]:
symbol = "AMZN"

In [6]:
ti = get_adjusted_intraday_by_sym(symbol=symbol, all_div=ALL_DIV, hols=US_HOLS, keep_unadj=True)

[JUSTY.LOG]	2026-02-18 19:21:57,252 - qt.common.help - INFO - df.shape = (1, 2)
[JUSTY.LOG]	2026-02-18 19:21:57,573 - qt.common.help - INFO - df.shape = (991766, 7)


In [19]:
ti

,timestamp,gmtoffset,open,high,low,close,volume,datetime,datetime_us,date,time
0,1609459260,0,"3,256.00","3,256.00","3,256.00","3,256.00",125,2021-01-01 00:01:00+00:00,2020-12-31 19:01:00-05:00,2020-12-31,19:01:00
1,1609459320,0,"3,255.40","3,255.40","3,255.40","3,255.40",126,2021-01-01 00:02:00+00:00,2020-12-31 19:02:00-05:00,2020-12-31,19:02:00
2,1609461360,0,"3,254.00","3,254.00","3,254.00","3,254.00",230,2021-01-01 00:36:00+00:00,2020-12-31 19:36:00-05:00,2020-12-31,19:36:00
3,1609462260,0,"3,252.61","3,252.61","3,252.61","3,252.61",120,2021-01-01 00:51:00+00:00,2020-12-31 19:51:00-05:00,2020-12-31,19:51:00
4,1609462740,0,"3,254.50","3,254.50","3,254.50","3,254.50",293,2021-01-01 00:59:00+00:00,2020-12-31 19:59:00-05:00,2020-12-31,19:59:00
...,...,...,...,...,...,...,...,...,...,...,...
991761,1770249360,0,233.77,233.85,233.51,233.79,6454,2026-02-04 23:56:00+00:00,2026-02-04 18:56:00-05:00,2026-02-04,18:56:00
991762,1770249420,0,233.52,233.85,233.51,233.85,404,2026-02-04 23:57:00+00:00,2026-02-04 18:57:00-05:00,2026-02-04,18:57:00
991763,1770249480,0,233.69,233.85,233.51,233.63,276,2026-02-04 23:58:00+00:00,2026-02-04 18:58:00-05:00,2026-02-04,18:58:00
991764,1770249540,0,233.65,233.85,233.63,233.83,406,2026-02-04 23:59:00+00:00,2026-02-04 18:59:00-05:00,2026-02-04,18:59:00


In [7]:
all_div = ALL_DIV
hols = US_HOLS
keep_unadj = True

In [8]:
# read all dividends data
assert hols is not None, f"US holidays data missing"
assert all_div is not None, f"dividend data is missing"

# get intraday and split data
ts = get_data_by_symbol_filename(symbol=symbol, filename='split')
ti = get_data_by_symbol_filename(symbol=symbol, filename='intraday')

# convert date to date type
if len(ts) > 0:
	ts['date'] = pd.to_datetime(ts['date']).dt.date	

# add datetime column from unix timestamp for raw unadjusted data
ti = add_dt_us_intraday(ti)
ti = ti.sort_values('datetime_us').reset_index(drop=True)
ti['close'] = ti.groupby('date')['close'].ffill()

[JUSTY.LOG]	2026-02-18 19:22:10,822 - qt.common.help - INFO - df.shape = (1, 2)
[JUSTY.LOG]	2026-02-18 19:22:11,063 - qt.common.help - INFO - df.shape = (991766, 7)


In [9]:
# create date, time lattice
start_date_sym	= ti['date'].min()
end_date_sym	= ti['date'].max()
start_time_sym	= dt.time(4, 0) # ti['time'].min()
end_time_sym	= dt.time(19, 59) # ti['time'].max()

# date and time lattice
dates = pd.bdate_range(start=start_date_sym, end=end_date_sym, freq="D").date
times = pd.date_range(
	start=dt.datetime.combine(dt.date.today(), start_time_sym),
	end=dt.datetime.combine(dt.date.today(), end_time_sym),
	freq="1min"
).time

# cartesian product
lattice = pd.MultiIndex.from_product(
	[dates, times],
	names=["date", "time"]
).to_frame(index=False)

In [16]:
pd.merge(
	pd.DataFrame(ti.groupby('date')['time'].min()),
	pd.DataFrame(ti.groupby('date')['time'].max()),
	on='date'
)

,time_x,time_y
date,,
2020-12-31,19:01:00,19:59:00
2021-01-04,04:42:00,19:54:00
2021-01-05,04:29:00,19:57:00
2021-01-06,04:00:00,19:53:00
2021-01-07,04:29:00,19:59:00
...,...,...
2026-01-29,04:00:00,19:59:00
2026-01-30,04:00:00,19:59:00
2026-02-02,04:00:00,19:59:00


In [ ]:
# do a left merge
til = pd.merge(lattice, ti, on=['date', 'time'], how='left')

# ffill close by date
til['close'] = til.groupby('date')['close'].ffill()


# keep close using prices in trading hours
til_c = pd.DataFrame(til.groupby('date')['close'].last()).reset_index().sort_values('date').reset_index(drop=True)
til_c['is_holiday'] = til_c['date'].apply(lambda x: x in hols)
til_c['is_weekend'] = pd.to_datetime(til_c['date']).dt.weekday >= 5

# drop weekends
til_c = til_c[~til_c['is_weekend']].sort_values('date').reset_index(drop=True)
til_c.drop(columns=['is_weekend'], inplace=True)

# forward fill close
til_c['close'] = til_c['close'].ffill()

# parse split ratio
if len(ts) > 0:
	ts["split_ratio"] = ts["split"].apply(lambda x: float(x.split("/")[1])/float(x.split("/")[0]))
	til_c = pd.merge(til_c, ts[['date', 'split_ratio']], on='date', how='left')
else:
	til_c['split_ratio'] = np.nan

# add split ratio and dividend data
tdiv_sym = all_div[all_div['Code'] == symbol].rename({'Date' : 'date', 'Dividend' : 'div_usd'}, axis=1)[['date', 'div_usd']]
if len(tdiv_sym) > 0:
	til_c = pd.merge(til_c, tdiv_sym, on='date', how='left')
else:
	til_c['div_usd'] = np.nan

# shift div and split adjustment prior to ex-date
til_c['div_usd_pre_ex'] = til_c['div_usd'].shift(-1)
til_c['div_adj'] = 1-(til_c['div_usd_pre_ex']/til_c['close'])
til_c['split_adj'] = til_c['split_ratio'].shift(-1)

# sort date
til_c = til_c.sort_values('date', ascending=False).reset_index(drop=True)

# cumulative split and div adjustments
til_c['split_adj_cum'] = til_c['split_adj'].fillna(1).cumprod()
til_c['div_adj_cum'] = til_c['div_adj'].fillna(1).cumprod()

# cumulative adjustments
til_c['adj_cum'] = til_c['split_adj_cum'] * til_c['div_adj_cum']

# adjust close
til_c['close_adj'] = til_c['close']*til_c['adj_cum']

# add log and raw returns
til_c["log_ret_1d"] = 100 * np.log(til_c["close_adj"]/til_c["close_adj"].shift(-1))
til_c["ret_1d"] = 100 * (-1 + (til_c["close_adj"]/til_c["close_adj"].shift(-1)))

# add adjustment factor to the lattice
til = pd.merge(til, til_c[['date', 'split_adj_cum', 'div_adj_cum', 'adj_cum']], on='date', how='left')

# combine date and time column
til["datetime_us"] = pd.to_datetime(til["date"].astype(str) + " " + til["time"].astype(str))

# filter useful columns
til = til[['datetime_us', 'date', 'time', 'open', 'high', 'low', 'close', 'volume', 'split_adj_cum', 'div_adj_cum', 'adj_cum']]


In [ ]:
ti["ret_1m"] = 100* np.log(ti["close"] / ti.groupby("date")["close"].shift(1)).fillna(0)

In [ ]:
ti = ti[
	(ti['time'] >= dt.time(9, 30)) &
	(ti['time'] <= dt.time(16, 0))
]

In [ ]:
sdate = dt.date(2025, 12, 24)

In [ ]:
ti.groupby('date').agg(
	mean_ret = ("ret_1m", "mean"),
	std_ret = ("ret_1m", "std"),
	min_ret = ("ret_1m", "min"),
	max_ret = ("ret_1m", "max"),
	count_zero_returns = ("ret_1m", lambda x: (x==0).sum()),
	count_all = ("ret_1m", "count")
).plot(y=['mean_ret', 'max_ret', 'min_ret', 'std_ret'])

- adjusted intraday history

In [17]:
tu = pd.read_csv('data/spy_cst.csv')
tickers = tu[tu['Ticker'] != '-']['Ticker'].unique().tolist()

In [18]:
for sym in tickers:
	
	try:
		qt.log.info(f"querying for ticker : {sym}")

		csv_file_name = f'data_adjusted_by_symbol/{sym}.csv'
		if Path(csv_file_name).is_file():
			qt.log.info(f"file already exists. ignoring")
		else:
			ti_sym = get_adjusted_intraday_by_sym(symbol=sym, all_div=ALL_DIV, hols=US_HOLS, keep_unadj=True)
			qt.log.info(f"got {ti_sym.shape} rows for sym {sym}, saving to {csv_file_name}")
			ti_sym.to_csv(csv_file_name, index=False)
	
	except Exception as e:
		qt.log.warning(f"failed for ticker {sym}, ereor : {e}")

[JUSTY.LOG]	2026-02-18 19:24:10,397 - qt.common.help - INFO - querying for ticker : NVDA
[JUSTY.LOG]	2026-02-18 19:24:10,398 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-18 19:24:10,399 - qt.common.help - INFO - querying for ticker : AAPL
[JUSTY.LOG]	2026-02-18 19:24:10,399 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-18 19:24:10,400 - qt.common.help - INFO - querying for ticker : MSFT
[JUSTY.LOG]	2026-02-18 19:24:10,400 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-18 19:24:10,401 - qt.common.help - INFO - querying for ticker : AMZN
[JUSTY.LOG]	2026-02-18 19:24:10,401 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-18 19:24:10,401 - qt.common.help - INFO - querying for ticker : GOOGL
[JUSTY.LOG]	2026-02-18 19:24:10,402 - qt.common.help - INFO - file already exists. ignoring
[JUSTY.LOG]	2026-02-18 19:24:10,402 - qt.common.help - INFO - querying for ticker : GOOG
[JUST

In [ ]:
ti_sym.groupby('date')['close'].apply(lambda x: (~x.isna()).sum())

In [ ]:
ti_sym.groupby('date')['close'].apply(lambda x: x.count())

In [ ]:
ti_sym[ti_sym['']]